# SmolVLM2 TinyDoc VISION QLoRA (Kaggle)

Vision-track run: `HuggingFaceTB/SmolVLM2-2.2B-Instruct` + LoRA on language AND SigLIP tower (out_proj/fc1/fc2; q/k/v match both) + unfrozen multi-modal projector, on the 1919-doc distill mix (419 3B-teacher + 1500 synth). Frozen val-40/test-100 NEVER leave the Mac; kernel eval split comes from train only. Checkpoints sync to Hub each save; final judgment is local clean-100.

In [ ]:
import subprocess, sys, os, time

# Single GPU only: DataParallel (2xT4) is broken with PEFT (StopIteration
# in SmolVLMModel.dtype on replicas). T4 has no bf16 -> script uses fp16.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

REPO_URL = 'https://github.com/eulogik/TinyDoc-VLM'
REPO = '/kaggle/working/tinydoc-vlm'

DATA_REPO = os.environ.get('DATA_REPO', 'eulogik/TinyDoc-VLM-distill')
HUB_ID = os.environ.get('HUB_ID', 'eulogik/SmolVLM2-TinyDoc-distill-vision')
MAX_STEPS = os.environ.get('MAX_STEPS', '300')
BATCH = os.environ.get('BATCH', '2')
ACCUM = os.environ.get('ACCUM', '8')
LR = os.environ.get('LR', '0.0002')
EVAL_EVERY = os.environ.get('EVAL_EVERY', '60')

if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        _c = UserSecretsClient()
        for _ in range(5):
            try:
                os.environ['HF_TOKEN'] = _c.get_secret('HF_TOKEN')
                break
            except Exception:
                time.sleep(5)
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN not set. Add it to Kaggle Secrets (Settings > Secrets).')

if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO])

# SmolVLM2 QLoRA stack (Kaggle base lacks trl/peft/bitsandbytes)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.50', 'trl>=0.14', 'peft', 'bitsandbytes',
    'pillow', 'datasets', 'huggingface_hub', 'num2words'])

p = subprocess.run(
    [sys.executable, 'training/smolvlm2_qlora.py',
     '--data-repo', DATA_REPO, '--hub-id', HUB_ID,
     '--max-steps', MAX_STEPS, '--batch', BATCH,
     '--accum', ACCUM, '--lr', LR,
     '--eval-every', EVAL_EVERY,
     '--vision-lora', '--train-projector',
     '--output-dir', '/kaggle/working/smolvlm2-output',
     '--model-path', 'HuggingFaceTB/SmolVLM2-2.2B-Instruct'],
    cwd=REPO,
)
sys.exit(p.returncode)

